[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/next-gen-aiam/blob/main/notebooks/06a_rl_reinforce.ipynb)

# Reinforcement Learning for Portfolio Construction I — REINFORCE on the 29-Asset Universe

REINFORCE+baseline on the full 29-asset universe with monthly walk-forward refit.  
Two configs: λ_risk = 0.02 (baseline) vs λ_risk = 0.00 (no risk penalty).  
Key question: does full-rollout training at N=29 produce feature-conditional policies, or does
the static collapse observed at N=1/2/3 with 60-step truncation persist at scale?

## Context: reinforcement learning as the fourth complexity test

This notebook is the **primary RL chapter** of Paper 2. It belongs to a sequence:

| Notebook | Paradigm | Best OOS Sharpe |
|---|---|---|
| [03](03_ml_strategies.ipynb) | ML ensemble (Lasso/RF/XGB → MSR) | **2.579** ← bar |
| [04](04_dl_strategies.ipynb) | DL predict-then-optimize | 2.386 |
| [05](05_dl_portfolio_construction_exploration.ipynb) / [05b](05b_cross_asset_deep_momentum.ipynb) | DL direct-weight (BSV 2009) | 1.240 / 0.852 |
| **06a** | **RL — REINFORCE** | **this notebook** |
| [06b](06b_rl_ppo.ipynb) | RL — PPO | confirmation run |

**The question:** can a reinforcement learning agent learn a *dynamic, feature-conditional* policy on the full 29-asset simplex — weights that vary meaningfully with market state — or does it collapse to a near-static allocation regardless of the observation?

**The finding (foregrounded):** REINFORCE **STATIC_COLLAPSE_DETECTED**. Both configurations (λ_risk = 0.02 and λ_risk = 0.00) converge to ≈1/N equal-weight; turnover and weight standard deviation fall below the collapse gate on every refit. OOS Sharpe ≈ 2.03 (rank ~26/40), well below the ML bar of 2.579 and the classical OOS bar of 2.422. This is not mere underperformance — it is **failure to learn a dynamic policy at all**, the fourth and sharpest leg of the cross-paradigm 'complexity doesn't pay' finding.

> The negative result is the result. A null on the most favorable RL setting (full rollouts, N=29 cross-asset universe, two λ ablations) is a rigorous, reproducible finding.

In [ ]:
!nvidia-smi

In [ ]:
import sys, os, time, warnings
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    print('Detected Colab — cloning repo...')
    os.system('git clone https://github.com/FranQuant/next-gen-aiam.git 2>&1 | tail -3')
    os.chdir('next-gen-aiam')
    print(f'Working dir: {os.getcwd()}')
    print('Installing dependencies...')
    os.system('pip install -q -e . 2>&1 | tail -3')
    os.system('pip install -q pyarrow scipy scikit-learn cvxpy pandas matplotlib seaborn torch 2>&1 | tail -3')
else:
    print(f'Detected local environment — {os.getcwd()}')

import torch

if torch.cuda.is_available():
    DEVICE = 'cuda'
    GPU_NAME = torch.cuda.get_device_name(0)
    if IS_COLAB:
        os.system('nvidia-smi')
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    GPU_NAME = 'Apple Silicon MPS'
else:
    DEVICE = 'cpu'
    GPU_NAME = 'CPU'

print(f'Device : {DEVICE}  ({GPU_NAME})')
print(f'torch  : {torch.__version__}')

# Run mode: full on Colab GPU, smoke on local CPU
if DEVICE == 'cuda':
    SEEDS     = list(range(10))
    EPISODES  = 200
    MAX_STEPS = None    # full rollouts (~500 steps per training episode)
    RUN_MODE  = 'full'
else:
    SEEDS     = [0]
    EPISODES  = 5
    MAX_STEPS = 30      # truncated for fast local smoke test
    RUN_MODE  = 'smoke'

print(f'Run mode : {RUN_MODE}')
print(f'Seeds    : {SEEDS}')
print(f'Episodes : {EPISODES}')
print(f'Max steps: {MAX_STEPS}')

## §1 Imports

Standard scientific stack, `_shared` helpers, and the `aiam.rl` walk-forward engine.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.2,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})
warnings.filterwarnings('ignore')

ROOT = Path(os.getcwd())
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent

for p in [str(ROOT / 'src'), str(ROOT / 'notebooks')]:
    if p not in sys.path:
        sys.path.insert(0, p)

from _shared import ann_sharpe, ann_return, ann_vol, max_drawdown, TRADING_DAYS
from aiam.data.split import TEST_START, TRAIN_END
from aiam.dl.walkforward import generate_refit_dates
from aiam.rl.env import PortfolioEnv
from aiam.rl.policy import SimplexPolicy
from aiam.rl.trainer import TrainConfig
from aiam.rl.walkforward import WalkForwardRLEnsemble, fit_walkforward_rl

RESULTS_DIR = ROOT / 'results' / 'rl' / 'n29'
FIG_DIR     = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

COST_BPS   = 10.0
LOOKBACK   = 20    # feature dimension = rolling 20-day return windows per asset
HIDDEN_DIM = 32
TRAIN_WINDOW_MONTHS = 24

print(f'ROOT       : {ROOT}')
print(f'Results dir: {RESULTS_DIR}')
print(f'TEST_START : {TEST_START}   TRAIN_END: {TRAIN_END}')

## §2 Data Loading — 29-Asset Returns

Load the canonical 29-asset daily returns cache and split at the 2022-12-31 train/test boundary.

In [ ]:
# Compute daily returns from prices
prices = pd.read_parquet(ROOT / 'data' / 'cache' / 'prices_29.parquet')
returns_all = prices.pct_change().dropna()

# Split
returns_train = returns_all.loc[:TRAIN_END]
returns_oos   = returns_all.loc[TEST_START:]
ASSETS = returns_all.columns.tolist()
N_ASSETS = len(ASSETS)

print(f'Universe   : {N_ASSETS} assets')
print(f'Full period: {returns_all.index[0].date()} → {returns_all.index[-1].date()}  ({len(returns_all)} days)')
print(f'Train      : {returns_train.index[0].date()} → {returns_train.index[-1].date()}  ({len(returns_train)} days)')
print(f'OOS        : {returns_oos.index[0].date()} → {returns_oos.index[-1].date()}   ({len(returns_oos)} days)')
print(f'\nAssets: {ASSETS}')

## §3 Walk-Forward Setup

Monthly refit cadence over the 2023–2026 OOS window — one REINFORCE+baseline fit per seed per rebalance date.

In [ ]:
oos_start = pd.Timestamp(TEST_START)
oos_end   = returns_all.index[-1]

refit_dates = generate_refit_dates(
    test_start=oos_start,
    test_end=oos_end,
    cadence='monthly',
    calendar=returns_all.index,
)

print(f'Refit dates: {len(refit_dates)}')
print(f'First refit: {refit_dates[0].date()}')
print(f'Last refit : {refit_dates[-1].date()}')
print()

# Show training windows for first 3 refits
for rd in refit_dates[:3]:
    train_end   = rd - pd.Timedelta(days=1)
    train_start = rd - pd.DateOffset(months=TRAIN_WINDOW_MONTHS)
    window_returns = returns_all.loc[
        (returns_all.index >= train_start) & (returns_all.index <= train_end)
    ]
    print(f'  Refit {rd.date()}: train {train_start.date()} → {train_end.date()} ({len(window_returns)} days)')

total_fits = 2 * len(SEEDS) * len(refit_dates)
print(f'\nTotal policy fits: 2 configs × {len(SEEDS)} seeds × {len(refit_dates)} refits = {total_fits}')

## §4 Training — Config A  (λ_risk = 0.02, REINFORCE baseline)

λ_risk = 0.02 adds a portfolio-variance penalty to the REINFORCE reward. This is the primary configuration.

In [ ]:
LAMBDA_A = 0.02

config_A = TrainConfig(
    episodes=EPISODES,
    gamma=0.95,
    lr=1e-3,
    entropy_coef=0.01,
    grad_clip=1.0,
    seed=0,
    max_steps_per_episode=MAX_STEPS,
)

print(f'Config A: lambda_risk={LAMBDA_A}, episodes={EPISODES}, max_steps={MAX_STEPS}')
t0 = time.time()

wf_A = fit_walkforward_rl(
    returns=returns_all,
    refit_dates=refit_dates,
    config=config_A,
    seeds=SEEDS,
    hidden_dim=HIDDEN_DIM,
    training_window_months=TRAIN_WINDOW_MONTHS,
    lambda_risk=LAMBDA_A,
    verbose=True,
)

elapsed_A = time.time() - t0
print(f'\nConfig A done in {elapsed_A:.1f}s ({elapsed_A/60:.1f} min)')

## §5 Training — Config B  (λ_risk = 0.00, no risk penalty)

Ablation: remove the risk penalty entirely to test whether it is the cause of collapse.

In [ ]:
LAMBDA_B = 0.00

config_B = TrainConfig(
    episodes=EPISODES,
    gamma=0.95,
    lr=1e-3,
    entropy_coef=0.01,
    grad_clip=1.0,
    seed=0,
    max_steps_per_episode=MAX_STEPS,
)

print(f'Config B: lambda_risk={LAMBDA_B}, episodes={EPISODES}, max_steps={MAX_STEPS}')
t0 = time.time()

wf_B = fit_walkforward_rl(
    returns=returns_all,
    refit_dates=refit_dates,
    config=config_B,
    seeds=SEEDS,
    hidden_dim=HIDDEN_DIM,
    training_window_months=TRAIN_WINDOW_MONTHS,
    lambda_risk=LAMBDA_B,
    verbose=True,
)

elapsed_B = time.time() - t0
print(f'\nConfig B done in {elapsed_B:.1f}s ({elapsed_B/60:.1f} min)')

## §6 OOS Evaluation

Assemble the walk-forward weight series into daily portfolio returns and compute Sharpe, return, vol, and drawdown.

In [ ]:
print('Evaluating Config A OOS...')
ret_A, weights_A, diag_A = wf_A.evaluate_oos(
    returns_all, oos_start=oos_start, oos_end=oos_end, lookback=LOOKBACK
)

print('Evaluating Config B OOS...')
ret_B, weights_B, diag_B = wf_B.evaluate_oos(
    returns_all, oos_start=oos_start, oos_end=oos_end, lookback=LOOKBACK
)

sharpe_A = ann_sharpe(ret_A)
sharpe_B = ann_sharpe(ret_B)

print(f'\nConfig A (λ=0.02): Sharpe={sharpe_A:.4f}  AnnRet={ann_return(ret_A):.4f}  AnnVol={ann_vol(ret_A):.4f}  MaxDD={max_drawdown(ret_A):.4f}')
print(f'Config B (λ=0.00): Sharpe={sharpe_B:.4f}  AnnRet={ann_return(ret_B):.4f}  AnnVol={ann_vol(ret_B):.4f}  MaxDD={max_drawdown(ret_B):.4f}')
print(f'\nML bar (MSR Ensemble): 2.579')
print(f'Gap A vs ML bar: {sharpe_A - 2.579:.3f}')
print(f'Gap B vs ML bar: {sharpe_B - 2.579:.3f}')

## §7 Static Collapse Gate

**GATE**: if `mean_turnover < 0.05` AND `mean_weight_std_across_time < 0.01` for BOTH configs,
the static collapse has reproduced at N=29 with full rollouts.

In [ ]:
TO_THRESH   = 0.05   # turnover threshold
WSTD_THRESH = 0.01   # weight std-across-time threshold

mean_to_A   = diag_A['mean_turnover']
wstd_A      = diag_A['weight_std_across_time']
mean_to_B   = diag_B['mean_turnover']
wstd_B      = diag_B['weight_std_across_time']

# Also compute within-refit training turnover from histories
train_to_A = np.mean([
    np.mean(h.mean_turnovers[-10:]) if h.mean_turnovers else 0.0
    for rr in wf_A.refit_results for h in rr.histories
])
train_to_B = np.mean([
    np.mean(h.mean_turnovers[-10:]) if h.mean_turnovers else 0.0
    for rr in wf_B.refit_results for h in rr.histories
])

collapsed_A = (mean_to_A < TO_THRESH) and (wstd_A < WSTD_THRESH)
collapsed_B = (mean_to_B < TO_THRESH) and (wstd_B < WSTD_THRESH)

print('=' * 60)
print('FEATURE-CONDITIONAL CHECK')
print('=' * 60)
print(f'Config A (λ=0.02)')
print(f'  OOS mean_turnover             : {mean_to_A:.5f}  (thresh {TO_THRESH})')
print(f'  OOS weight_std_across_time    : {wstd_A:.5f}  (thresh {WSTD_THRESH})')
print(f'  Training mean_turnover (last10): {train_to_A:.5f}')
print(f'  Static collapse?               : {collapsed_A}')
print()
print(f'Config B (λ=0.00)')
print(f'  OOS mean_turnover             : {mean_to_B:.5f}  (thresh {TO_THRESH})')
print(f'  OOS weight_std_across_time    : {wstd_B:.5f}  (thresh {WSTD_THRESH})')
print(f'  Training mean_turnover (last10): {train_to_B:.5f}')
print(f'  Static collapse?               : {collapsed_B}')
print()

if collapsed_A and collapsed_B:
    verdict = 'STATIC_COLLAPSE_DETECTED'
    print('*** STATIC_COLLAPSE_DETECTED ***')
    print('Both configs collapsed to static policies at N=29 with full rollouts.')
    print('This IS the finding. Proceeding to comparison table and findings doc.')
elif collapsed_A or collapsed_B:
    verdict = 'PARTIAL_COLLAPSE'
    which = 'Config A' if collapsed_A else 'Config B'
    print(f'*** PARTIAL_COLLAPSE ({which} collapsed, other did not) ***')
else:
    verdict = 'FEATURE_CONDITIONAL_LEARNING'
    print('*** FEATURE_CONDITIONAL_LEARNING ***')
    print('At least one config shows non-trivial turnover and weight variation.')
    print('Full rollouts at N=29 enabled feature-conditional learning.')

print(f'\nVERDICT: {verdict}')

## §8 Training Diagnostics

Per-refit reward curves and policy entropy — visualize whether the agent is learning or stagnating.

In [ ]:
# Build per-refit-per-seed diagnostics
diag_rows = []
for config_name, wf_ensemble, lambda_val in [
    ('config_A', wf_A, LAMBDA_A),
    ('config_B', wf_B, LAMBDA_B),
]:
    for rr in wf_ensemble.refit_results:
        for seed_idx, (agent, hist) in enumerate(zip(rr.agents, rr.histories)):
            seed = SEEDS[seed_idx]
            final_reward = hist.episode_rewards[-1] if hist.episode_rewards else float('nan')
            mean_to_train = float(np.mean(hist.mean_turnovers)) if hist.mean_turnovers else float('nan')
            last10_to     = float(np.mean(hist.mean_turnovers[-10:])) if hist.mean_turnovers else float('nan')
            mean_w = hist.mean_weights[-1] if hist.mean_weights else np.full(N_ASSETS, float('nan'))
            w_std  = float(np.std(mean_w)) if not np.all(np.isnan(mean_w)) else float('nan')
            diag_rows.append({
                'config': config_name,
                'lambda_risk': lambda_val,
                'refit_date': rr.refit_date,
                'seed': seed,
                'final_episode_reward': final_reward,
                'mean_turnover_training': mean_to_train,
                'last10ep_turnover': last10_to,
                'final_weight_std': w_std,
            })

diag_df = pd.DataFrame(diag_rows)
diag_path = RESULTS_DIR / 'diagnostics_all.parquet'
diag_df.to_parquet(diag_path)
print(f'Saved diagnostics → {diag_path}')
print(diag_df.groupby('config')[['mean_turnover_training','last10ep_turnover','final_weight_std']].mean().round(5).to_string())

## §9 Comparison vs 37-Strategy Table

Merge RL results into the full cross-paradigm leaderboard and report rank.

In [ ]:
base_csv = ROOT / 'data' / 'cache' / 'portfolio_returns' / 'dl_strategies_extended_comparison.csv'
base_df  = pd.read_csv(base_csv, index_col=0)
print(f'Base comparison: {len(base_df)} strategies')

# EW benchmark from base returns for turnover/tc stats
def _rl_row(ret_series: pd.Series, config_label: str, family: str) -> dict:
    return {
        'Ann Ret' : ann_return(ret_series),
        'Ann Vol' : ann_vol(ret_series),
        'Sharpe'  : ann_sharpe(ret_series),
        'Max DD'  : max_drawdown(ret_series),
        'Family'  : family,
    }

rl_rows = pd.DataFrame([
    _rl_row(ret_A, 'RL_lambda_002_ensemble', 'RL (REINFORCE, λ=0.02)'),
    _rl_row(ret_B, 'RL_lambda_000_ensemble', 'RL (REINFORCE, λ=0.00)'),
], index=['RL_lambda_002_ensemble', 'RL_lambda_000_ensemble'])

full_df = pd.concat([base_df, rl_rows])
full_df = full_df.sort_values('Sharpe', ascending=False)

out_csv = ROOT / 'data' / 'cache' / 'portfolio_returns' / 'full_comparison_with_rl.csv'
full_df.to_csv(out_csv)
print(f'Full comparison ({len(full_df)} rows) → {out_csv}')

# Top 10 highlight
print('\nTop 10 by net Sharpe:')
top10 = full_df.head(10)
for i, (strat, row) in enumerate(top10.iterrows(), 1):
    marker = ' <-- RL' if 'RL_lambda' in strat else ''
    print(f'  {i:2d}. {strat:<45} Sharpe={row["Sharpe"]:.4f}  Family={row["Family"]}{marker}')

## §10 Per-Config Summary Statistics

Sharpe, return, vol, max drawdown, turnover, and weight concentration for each configuration.

In [ ]:
# Per-seed Sharpe across refits (for seed stability metric)
# We approximate per-seed Sharpe by re-running OOS for each seed's agents
# This is diagnostic — may be slow; skip in smoke mode
if RUN_MODE == 'full':
    print('Computing per-seed Sharpe distribution (Config A)...')
    from aiam.rl.walkforward import WalkForwardRLEnsemble, RLRefitResult
    seed_sharpes_A, seed_sharpes_B = [], []

    for seed_idx in range(len(SEEDS)):
        # Build single-seed ensemble for Config A
        single_A = WalkForwardRLEnsemble(refit_results=[
            RLRefitResult(
                refit_date=rr.refit_date,
                agents=[rr.agents[seed_idx]],
                histories=[rr.histories[seed_idx]],
            )
            for rr in wf_A.refit_results
        ])
        r_A, _, _ = single_A.evaluate_oos(returns_all, oos_start, oos_end, lookback=LOOKBACK)
        seed_sharpes_A.append(ann_sharpe(r_A))

        single_B = WalkForwardRLEnsemble(refit_results=[
            RLRefitResult(
                refit_date=rr.refit_date,
                agents=[rr.agents[seed_idx]],
                histories=[rr.histories[seed_idx]],
            )
            for rr in wf_B.refit_results
        ])
        r_B, _, _ = single_B.evaluate_oos(returns_all, oos_start, oos_end, lookback=LOOKBACK)
        seed_sharpes_B.append(ann_sharpe(r_B))

    print(f'Config A: Sharpe {np.mean(seed_sharpes_A):.4f} ± {np.std(seed_sharpes_A):.4f}  (seeds: {[f"{s:.3f}" for s in seed_sharpes_A]})')
    print(f'Config B: Sharpe {np.mean(seed_sharpes_B):.4f} ± {np.std(seed_sharpes_B):.4f}  (seeds: {[f"{s:.3f}" for s in seed_sharpes_B]})')

    # Save seed-level results
    seed_df = pd.DataFrame({
        'seed': SEEDS,
        'sharpe_config_A': seed_sharpes_A,
        'sharpe_config_B': seed_sharpes_B,
    })
    seed_df.to_csv(RESULTS_DIR / 'seed_sharpe_distribution.csv', index=False)
    print(f'Saved → {RESULTS_DIR}/seed_sharpe_distribution.csv')
else:
    print('Smoke mode: skipping per-seed breakdown.')
    seed_sharpes_A = [sharpe_A]
    seed_sharpes_B = [sharpe_B]

## §11 Figures

Equity curves, drawdowns, and weight heatmaps across the OOS walk-forward window.

In [ ]:
# --- Figure 1: equity_curves_rl.png ---
# OOS cumulative returns: RL configs + top-5 comparison strategies

# Load OOS returns for top-5 comparison strategies
# (from the 31-strategy parquet — use as gross proxy; net handled in CSV)
base_rets_path = ROOT / 'data' / 'cache' / 'portfolio_returns' / '31strategies_29assets_2003_2026.parquet'
try:
    base_rets = pd.read_parquet(base_rets_path)
    base_rets_oos = base_rets.loc[TEST_START:]
    top5_strats = full_df[~full_df.index.str.contains('RL_lambda')].head(5).index.tolist()
    available_strats = [s for s in top5_strats if s in base_rets_oos.columns]
    compare_rets = base_rets_oos[available_strats[:3]] if available_strats else pd.DataFrame()
except Exception:
    compare_rets = pd.DataFrame()

fig, ax = plt.subplots(figsize=(12, 5))

# RL curves
cum_A = (1 + ret_A).cumprod()
cum_B = (1 + ret_B).cumprod()
ax.plot(cum_A.index, cum_A.values, label=f'RL λ=0.02 (Sharpe {sharpe_A:.3f})', lw=2, color='steelblue')
ax.plot(cum_B.index, cum_B.values, label=f'RL λ=0.00 (Sharpe {sharpe_B:.3f})', lw=2, color='tomato')

# Comparison curves
colors = ['#888', '#aaa', '#bbb']
for i, col in enumerate(compare_rets.columns):
    cum = (1 + compare_rets[col]).cumprod()
    ax.plot(cum.index, cum.values, label=col, lw=1, alpha=0.7, color=colors[i % len(colors)])

ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return')
ax.set_title('OOS Equity Curves — RL (REINFORCE) vs Comparison Strategies')
ax.legend(loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'equity_curves_rl.png', dpi=150)
plt.show()
print(f'Saved equity_curves_rl.png')

In [ ]:
# --- Figure 2: turnover_over_time_rl.png ---

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, weights_df, config_name, lambda_val in [
    (axes[0], weights_A, 'Config A', LAMBDA_A),
    (axes[1], weights_B, 'Config B', LAMBDA_B),
]:
    # Daily turnover: sum of absolute weight changes
    to = weights_df.diff().abs().sum(axis=1).fillna(0)
    # Rebase to monthly for readability
    to_monthly = to.resample('ME').mean()
    ax.bar(to_monthly.index, to_monthly.values, width=20, alpha=0.7)
    ax.axhline(TO_THRESH, color='red', lw=1, ls='--', label=f'threshold={TO_THRESH}')
    ax.set_title(f'{config_name} (λ={lambda_val}) | mean={to.mean():.4f}')
    ax.set_xlabel('Date')
    ax.set_ylabel('Turnover')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle('OOS Daily Turnover (Monthly Average)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'turnover_over_time_rl.png', dpi=150)
plt.show()
print('Saved turnover_over_time_rl.png')

In [ ]:
# --- Figure 3: weight_heatmap_rl.png ---

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, weights_df, config_name, lambda_val in [
    (axes[0], weights_A, 'Config A', LAMBDA_A),
    (axes[1], weights_B, 'Config B', LAMBDA_B),
]:
    # Resample to monthly mean for readability
    w_monthly = weights_df.resample('ME').mean()
    im = ax.imshow(
        w_monthly.T.values,
        aspect='auto',
        cmap='RdYlGn',
        vmin=0,
        vmax=0.2,
    )
    ax.set_yticks(range(N_ASSETS))
    ax.set_yticklabels([a.split('.')[0] for a in ASSETS], fontsize=7)
    ax.set_xticks(range(0, len(w_monthly), max(1, len(w_monthly)//6)))
    ax.set_xticklabels([d.strftime('%Y-%m') for d in w_monthly.index[::max(1, len(w_monthly)//6)]], rotation=30, fontsize=8)
    ax.set_title(f'{config_name} (λ={lambda_val}) — Monthly Mean Weights')
    plt.colorbar(im, ax=ax, fraction=0.03)

fig.suptitle('Weight Heatmap Over Time (RL Ensemble)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'weight_heatmap_rl.png', dpi=150)
plt.show()
print('Saved weight_heatmap_rl.png')

In [ ]:
# --- Figure 4: seed_sharpe_dist_rl.png ---

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)

for ax, sharpe_list, config_name, lambda_val in [
    (axes[0], seed_sharpes_A, 'Config A', LAMBDA_A),
    (axes[1], seed_sharpes_B, 'Config B', LAMBDA_B),
]:
    ax.hist(sharpe_list, bins=max(3, len(sharpe_list)//2), edgecolor='k', alpha=0.7)
    ax.axvline(np.mean(sharpe_list), color='red', lw=2, label=f'mean={np.mean(sharpe_list):.3f}')
    ax.axvline(2.579, color='green', lw=1.5, ls='--', label='ML bar 2.579')
    ax.set_title(f'{config_name} (λ={lambda_val}) — Per-Seed Sharpe')
    ax.set_xlabel('Sharpe')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle('Per-Seed OOS Sharpe Distribution (RL REINFORCE, N=29)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'seed_sharpe_dist_rl.png', dpi=150)
plt.show()
print('Saved seed_sharpe_dist_rl.png')

## §12 Summary

Full-run output summary — printed after Colab execution completes.

In [ ]:
rl_rank_A = list(full_df.index).index('RL_lambda_002_ensemble') + 1
rl_rank_B = list(full_df.index).index('RL_lambda_000_ensemble') + 1

print('=' * 70)
print('RL REINFORCE SUMMARY')
print('=' * 70)
print(f'Universe    : {N_ASSETS} assets, 2003-2026')
print(f'Methodology : REINFORCE+baseline, monthly walk-forward, {len(refit_dates)} refits')
print(f'Configs     : 2 (λ=0.02, λ=0.00)  |  Seeds: {len(SEEDS)}  |  Episodes: {EPISODES}')
print(f'Max steps   : {MAX_STEPS} ({"full rollout" if MAX_STEPS is None else "truncated"})')
print(f'Total fits  : {2 * len(SEEDS) * len(refit_dates)}')
print()
print(f'Config A (λ=0.02): Sharpe={sharpe_A:.4f}  Rank={rl_rank_A}/{len(full_df)}')
print(f'Config B (λ=0.00): Sharpe={sharpe_B:.4f}  Rank={rl_rank_B}/{len(full_df)}')
print(f'ML bar (reference): 2.579  (MSR Ensemble)')
print()
print(f'Static collapse verdict: {verdict}')
print()
print('Artifacts saved:')
print(f'  {out_csv}')
print(f'  {RESULTS_DIR}/diagnostics_all.parquet')
print(f'  {FIG_DIR}/*.png')
print()
print('Next: docs/handoff/session_4c_findings.md (fill in actual numbers)')

In [ ]:
# Bundle artifacts for download
!cd /content/next-gen-aiam && zip -r /content/rl_n29_artifacts.zip \
    results/rl/n29/ docs/handoff/session_4c_findings.md

print("Ready to download: /content/rl_n29_artifacts.zip")
print("Open the file panel (folder icon, left sidebar) → find the zip → right-click → Download")

In [ ]:
# HTML export with all outputs preserved
!jupyter nbconvert --to html \
    /content/next-gen-aiam/notebooks/06a_rl_reinforce.ipynb \
    --output /content/06a_rl_reinforce.html

print("HTML snapshot: /content/06a_rl_reinforce.html — download from file panel")